## copys和views
在操作数组时，数组中的数据有时会被复制到一个新数组中，有时则不会。这常常会让初学者感到困惑。这种情况有三种：

### No copy at all（引用赋值（别名））
简单赋值不会复制对象或其数据。

In [2]:
import numpy as np
a = np.array([[ 0,  1,  2,  3],
              [ 4,  5,  6,  7],
              [ 8,  9, 10, 11]])
 # 注意：这里不是复制，只是创建了一个新的引用
b = a            # no new object is created
# 返回 True，表明 a 和 b 指向同一个内存对象
b is a           # a and b are two names for the same ndarray object

True

Python 将可变对象作为引用传递，因此函数调用不会复制对象。

In [4]:
def f(x):
    print(id(x)) # 打印传入对象 x 的唯一标识符
# 打印对象 a 的唯一标识符（假设 a 是之前定义的数组）
print(id(a))  # id is a unique identifier of an object 
f(a)   

1993548491504
1993548491504


### view或shallow copy（浅拷贝)
不同的数组对象可以共享相同的数据。该view方法会创建一个新的数组对象，该对象会查看相同的数据。

In [8]:
c = a.view() # c 是 a 的视图（共享数据，但可能拥有不同的元数据）
 # False → c 和 a 不是同一个对象
print (c is a)
 # True  → c 的数据基础（base）是 a
print (c.base is a )           # c is a view of the data owned by a
# False → c 不拥有自己的数据，只是引用 a 的数据
print (c.flags.owndata)
# c 被重新赋值，形状变为 (2,6)
c = c.reshape((2, 6))  # a's shape doesn't change, reassigned c is still a view of a
 # (3,4) → a 的形状不受影响
print (a.shape)
c[0, 4] = 1234        # a's data changes
print (a)

False
True
False
(3, 4)
[[   0    1    2    3]
 [1234    5    6    7]
 [   8    9   10   11]]


### Deep copy（深拷贝)
copy方法会完整复制数组及其数据。

In [10]:
d = a.copy()  # a new array object with new data is created
print (d is a)
print (d.base is a)  # d doesn't share anything with a
d[0, 0] = 9999
print (a)

False
False
[[   0    1    2    3]
 [1234    5    6    7]
 [   8    9   10   11]]


有时copy，如果不再需要原始数组，则应在切片后调用深拷贝。例如，假设a中间结果很大，而最终结果b只包含其中的一小部分，那么在使用切片a构造数组时，应该进行深拷贝：b

In [12]:
a = np.arange(int(1e8))
b = a[:100].copy()
del a  # the memory of ``a`` can be released.

## view vs copy总结

| 操作 | 共享数据 | `base` 属性 | `owndata` | 修改是否影响原数组 |
|------|----------|-------------|-----------|-------------------|
| `b = a` | ✅ | `a` | False | ✅ |
| `c = a.view()` | ✅ | `a` | False | ✅ |
| `d = a.copy()` | ❌ | `None` | True | ❌ |

## 重要结论
- **视图**：`c.base is a` 为 `True`，`c.flags.owndata` 为 `False`。
- **形状改变不影响原数组**：对视图 `reshape` 后，原数组形状保持不变。
- **数据修改影响原数组**：通过视图修改元素会直接改变原数组的数据。
- **视图索引映射**：视图的索引位置按行优先顺序映射到原数组的对应位置。